In [1]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/final_features.csv")
print("Shape:", df.shape)
df.head()

Shape: (9240, 31)


,Total Time Spent on Website,Page Views Per Visit,TotalVisits,What is your current occupation_Working Professional,Lead Origin_Lead Add Form,What matters most to you in choosing a course_Unknown,What is your current occupation_Unknown,Lead Source_Reference,What is your current occupation_Unemployed,Specialization_Unknown,...,Specialization_Finance Management,Specialization_Marketing Management,Country_Unknown,City_Other Metro Cities,Specialization_Operations Management,City_Other Cities of Maharashtra,Specialization_Business Administration,Specialization_Supply Chain Management,Country_Other,Converted
0,0,0.0,0.0,False,False,False,False,False,True,True,...,False,False,True,False,False,False,False,False,False,0
1,674,2.5,5.0,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,0
2,1532,2.0,2.0,False,False,False,False,False,False,False,...,False,False,False,False,False,False,True,False,False,1
3,305,1.0,1.0,False,False,False,False,False,True,False,...,False,False,False,False,False,False,False,False,False,0
4,1428,1.0,2.0,False,False,False,False,False,True,True,...,False,False,False,False,False,False,False,False,False,1


In [2]:
# Convert boolean columns to integers (0/1) for consistency
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].astype(int)

print("Converted boolean columns to int:", len(bool_cols))
df.head()

Converted boolean columns to int: 25


,Total Time Spent on Website,Page Views Per Visit,TotalVisits,What is your current occupation_Working Professional,Lead Origin_Lead Add Form,What matters most to you in choosing a course_Unknown,What is your current occupation_Unknown,Lead Source_Reference,What is your current occupation_Unemployed,Specialization_Unknown,...,Specialization_Finance Management,Specialization_Marketing Management,Country_Unknown,City_Other Metro Cities,Specialization_Operations Management,City_Other Cities of Maharashtra,Specialization_Business Administration,Specialization_Supply Chain Management,Country_Other,Converted
0,0,0.0,0.0,0,0,0,0,0,1,1,...,0,0,1,0,0,0,0,0,0,0
1,674,2.5,5.0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,0
2,1532,2.0,2.0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,0,0,1
3,305,1.0,1.0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,1428,1.0,2.0,0,0,0,0,0,1,1,...,0,0,0,0,0,0,0,0,0,1


In [3]:
X = df.drop(columns=['Converted'])
y = df['Converted']

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (9240, 30)
Target shape: (9240,)


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)
print("\nTraining target distribution:")
print(y_train.value_counts(normalize=True))
print("\nTest target distribution:")
print(y_test.value_counts(normalize=True))

Training set: (7392, 30)
Test set: (1848, 30)

Training target distribution:
Converted
0    0.614583
1    0.385417
Name: proportion, dtype: float64

Test target distribution:
Converted
0    0.614719
1    0.385281
Name: proportion, dtype: float64


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete ✅")

Scaling complete ✅


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_log = log_reg.predict(X_test_scaled)

print("Logistic Regression Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_log))
print("Precision:", precision_score(y_test, y_pred_log))
print("Recall:", recall_score(y_test, y_pred_log))
print("F1 Score:", f1_score(y_test, y_pred_log))
print("\nClassification Report:\n", classification_report(y_test, y_pred_log))

Logistic Regression Results:
Accuracy: 0.7916666666666666
Precision: 0.7711442786069652
Recall: 0.6530898876404494
F1 Score: 0.7072243346007605

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.88      0.84      1136
           1       0.77      0.65      0.71       712

    accuracy                           0.79      1848
   macro avg       0.79      0.77      0.77      1848
weighted avg       0.79      0.79      0.79      1848



In [7]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)  # Random Forest doesn't need scaled data

y_pred_rf = rf_model.predict(X_test)

print("Random Forest Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))

Random Forest Results:
Accuracy: 0.7797619047619048
Precision: 0.7447833065810594
Recall: 0.651685393258427
F1 Score: 0.6951310861423221

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.86      0.83      1136
           1       0.74      0.65      0.70       712

    accuracy                           0.78      1848
   macro avg       0.77      0.76      0.76      1848
weighted avg       0.78      0.78      0.78      1848



In [8]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss')
xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print("XGBoost Results:")
print("Accuracy:", accuracy_score(y_test, y_pred_xgb))
print("Precision:", precision_score(y_test, y_pred_xgb))
print("Recall:", recall_score(y_test, y_pred_xgb))
print("F1 Score:", f1_score(y_test, y_pred_xgb))
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))

XGBoost Results:
Accuracy: 0.7803030303030303
Precision: 0.7353846153846154
Recall: 0.6713483146067416
F1 Score: 0.7019089574155654

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.85      0.83      1136
           1       0.74      0.67      0.70       712

    accuracy                           0.78      1848
   macro avg       0.77      0.76      0.76      1848
weighted avg       0.78      0.78      0.78      1848



In [9]:
results = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [accuracy_score(y_test, y_pred_log), accuracy_score(y_test, y_pred_rf), accuracy_score(y_test, y_pred_xgb)],
    'Precision': [precision_score(y_test, y_pred_log), precision_score(y_test, y_pred_rf), precision_score(y_test, y_pred_xgb)],
    'Recall': [recall_score(y_test, y_pred_log), recall_score(y_test, y_pred_rf), recall_score(y_test, y_pred_xgb)],
    'F1 Score': [f1_score(y_test, y_pred_log), f1_score(y_test, y_pred_rf), f1_score(y_test, y_pred_xgb)]
})

print(results)

                 Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression  0.791667   0.771144  0.653090  0.707224
1        Random Forest  0.779762   0.744783  0.651685  0.695131
2              XGBoost  0.780303   0.735385  0.671348  0.701909
